# The ± pairing partners: a complete account

**What this notebook is.** A self-contained, didactic synthesis of the July 2026 investigation that started with "why do the temporal CFT formulas seem to want $v=2$ at every $p$?" and ended with a full explanation of the Alcaraz transfer-matrix spectrum — including the long-standing "$\pm$ eigenvalue pairing" mystery, the "missing" boundary states, and a sharper understanding of *why the block power method walls earlier for the frustrated model than for Ising*. **Every claim below is supported in place**: either by a code cell that regenerates the numbers from the same caches the working notebooks use, or by an explicit pointer to the notebook section that established it. The detailed experimental record is `NBs/7_temp_c.ipynb`, section "Stress-testing the $v_{\rm eff}\approx2$ conclusion" (Checks 1–8).

**The cast, in three sentences.** The transverse method turns the Loschmidt-echo network into a *spatial transfer matrix* $E$: an operator that shifts the whole time-slice sideways by one lattice site. Each eigenvalue $\lambda$ of $E$ has a magnitude (how fast that state's contribution decays with distance) and a phase (how fast it *oscillates* — literally the momentum of the state, in radians per site). At a critical point, CFT predicts the top of $E$'s spectrum: a "tower" of states whose phases differ from the leading eigenvalue $\lambda_0$ by tiny steps $\pi x/(vT)$, where the $x$ are universal scaling dimensions and $v$ is the model's sound velocity — so reading the phases is reading the CFT's operator content.

**Background.** The conformal-field-theory vocabulary used throughout (central charge $c$, scaling dimensions, the operator tower, boundary states, the transverse space↔time trick) is built from scratch in the companion `cft_primer.ipynb`; the *fermion* vocabulary specific to the mechanism here (Jordan–Wigner, Majoranas, parity, Kramers–Wannier duality) is the short **Interlude** after §3. A reader comfortable with the basics can go straight through; each unfamiliar term has a plain-language home.

## 1. The three symptoms

By mid-July 2026 the Alcaraz model ($p\neq0$) showed three seemingly unrelated anomalies that Ising ($p=0$) did not:

1. **The velocity puzzle.** The boundary-exponent formula $x_1 = v\,a_1/\pi$ gave the clean free-boundary value $x_1\approx1/2$ only when fed $v=2$ (the Ising velocity) — even though the model's true sound velocity grows sharply with $p$. The velocity measurement itself (NB7, section "The velocity grows with frustration": Alcaraz's own Eq. 19 on exact ring spectra, validated by reproducing his published $c(p)\approx0.5$ via his Eq. 18) gives $v(0)=1.98$, $v(0.1)=2.60$, $v(0.3)=3.80$, $v(0.5)=4.93$:

![the measured sound velocity v(p)](results/imgs/nb4_velocity_vs_p.png)

2. **Missing states.** At $p=0.1$ the $x=1/2$ boundary state was nowhere in the top-10 eigenvalues by modulus; the deep $k=8/10$ scans found no odd rung ($1/2, 3/2, 5/2$) at all (NB3 deep-block scan; reproduced in the tower table of §2 below).
3. **The ± pairing.** At any $p\neq0$, eigenvalues appeared $\approx\pi$ away in phase from $\lambda_0$, with moduli approaching $|\lambda_0|$ — absent at $p=0$ — sharpening monotonically with $T$ (documented at length in `next_steps.md`; quantified in §6 below).

The resolution: **all three are one phenomenon**, and unpacking it took two methodological tools plus one decisive control experiment.

## 2. The tools: how to read a spectrum without fooling yourself

**Tool 1 — the velocity-free ratio test.** Assigning a dimension $x = vT|\Delta\phi|/\pi$ to a phase gap requires assuming a velocity, and a wrong $v$ can be silently absorbed by matching to the wrong rung of the ladder. But *ratios* of gaps within one tower cancel both $v$ and $T$: different candidate ladders predict different ratios, so ratios identify states with no velocity input at all.

**Tool 2 — the XY anisotropy control** (NB7 Check 4). To test whether the pipeline itself reads velocities honestly, we ran it on the anisotropic XY chain: $c=1/2$ (same universality) but *exactly known, tunable* $v(\gamma)=2\gamma$. The cell below regenerates the verdict: the fitted $a_1$ tracks $\pi/(4\gamma)$ across $v=1,2,3$ — a pipeline that pinned $v\approx2$ would have returned $a_1\approx0.785$ three times. The machinery is *velocity-faithful*; any "$v=2$ preference" had to be a misreading of the data.

In [1]:
# Tool 2 regenerated: the XY velocity-faithfulness verdict (cache: nb7_xy_control.jld2, NB7 Check 4).
using JLD2, Printf, LinearAlgebra
ph(t) = angle(-t)                                   # Im(log(-τ)): the paper's phase convention
dphw(a, b) = mod(ph(a) - ph(b) + pi, 2pi) - pi      # wrapped phase gap

xy = load("results/data/nb7_xy_control.jld2", "done")
println("fitted a1 (gap = a1/T + a3/T³) vs the velocity-faithful prediction π/(4γ):")
for gamma in (0.5, 1.0, 1.5)
    Ts = Float64[]; gaps = Float64[]
    for T in sort([k[2] for k in keys(xy) if k[1] == gamma])
        e = xy[(gamma, T)]
        cand = [dphw(e.theta[j], e.theta_phys) for j in eachindex(e.theta)
                if j != e.i0 && abs(dphw(e.theta[j], e.theta_phys)) < pi/2]
        isempty(cand) && continue
        push!(Ts, T); push!(gaps, cand[argmin(abs.(cand))])
    end
    q = hcat(1.0 ./ Ts, 1.0 ./ Ts.^3) \ gaps
    a1 = abs(q[1])
    @printf("  γ=%.1f (v=%.0f):  a1 = %.4f   π/(4γ) = %.4f   [pinned-v=2 would give 0.7854]   ⇒ x1 = %.3f\n",
            gamma, 2gamma, a1, pi/(4gamma), 2gamma*a1/pi)
end

fitted a1 (gap = a1/T + a3/T³) vs the velocity-faithful prediction π/(4γ):


  γ=0.5 (v=1):  a1 = 1.4810   π/(4γ) = 1.5708   [pinned-v=2 would give 0.7854]   ⇒ x1 = 0.471
  γ=1.0 (v=2):  a1 = 0.7820   π/(4γ) = 0.7854   [pinned-v=2 would give 0.7854]   ⇒ x1 = 0.498


  γ=1.5 (v=3):  a1 = 0.5199   π/(4γ) = 0.5236   [pinned-v=2 would give 0.7854]   ⇒ x1 = 0.496


**With those tools, the $p=0.1$ tower resolves cleanly** (NB7 Check 2 + Check 6). The cell below regenerates the three exhibits side by side: (i) the free-BC $p=0$ baseline showing the *full* ladder including the odd rungs; (ii) the free-BC $p=0.1$ deep tower, which under the true $v(p)$ is the **identity module** $\{2,3,4\}$ — three rungs to $\le3.5\%$, ratios $\to\{1, 1.5, 2\}$, each rung independently implying $v\approx2.6$; and (iii) the fixed-BC $p=0$ tower — the cleanest identity-module template we have — which is structurally identical to (ii). The "$v=2$" reading of $p=0.1$ had matched a single rung ($1.48\approx3/2$) and ignored that its second state ($2.31$) fits nothing.

In [2]:
# Tower identification, regenerated (caches: nb3_gap_k6, nb3_p01_stress, nb7_alcaraz_fixedbc).
deep  = load("results/data/nb3_p01_stress.jld2", "done")
k6    = load("results/data/nb3_gap_k6.jld2", "done")
fixed = load("results/data/nb7_alcaraz_fixedbc.jld2", "done")
v_p0, v_p01 = 1.977, 2.601                          # measured spatial velocities (NB7)

tower_gaps(th) = begin
    ths = sort(collect(th), by=abs, rev=true)
    sort([abs(dphw(t, ths[1])) for t in ths[2:end] if abs(dphw(t, ths[1])) < pi/2])
end
show_x(gaps, T, v) = join([@sprintf("%.3f", v*T*g/pi) for g in gaps], ", ")
show_r(gaps) = join([@sprintf("%.3f", g/gaps[1]) for g in gaps], ", ")

g = tower_gaps(k6[(:k6, 0.0, 7.0)].theta)
println("(i)  free-BC p=0,  T=7 : x@v(0) = ", show_x(g, 7.0, v_p0), "   (full ladder: ½,3/2,2,5/2,3 — odd rungs PRESENT)")

g = tower_gaps(deep[(0.1, 4.0, 8)].theta)
println("(ii) free-BC p=0.1,T=4 : x@v(p) = ", show_x(g, 4.0, v_p01), "   ratios = ", show_r(g), "   (identity module {2,3,4}: ratios 1, 1.5, 2)")
implied = [pi*x/(4.0*gg) for (x, gg) in zip([2.0, 3.0, 4.0], g)]
println("      per-rung implied v = ", join([@sprintf("%.2f", vv) for vv in implied], ", "), "   (measured spatial v = 2.60)")
println("      the same tower read with v=2: ", show_x(g, 4.0, 2.0), "  — only the first value is near a rung")

g = tower_gaps(fixed[(0.0, 7.0)].theta)
println("(iii)fixed-BC p=0, T=7 : x@v(0) = ", show_x(g, 7.0, v_p0), "   ratios = ", show_r(g), "   (the χ₁ template — same structure as (ii))")

(i)  free-BC p=0,  T=7 : x@v(0) = 0.489, 1.469, 1.958, 2.468, 2.957

   (full ladder: ½,3/2,2,5/2,3 — odd rungs PRESENT)
(ii) free-BC p=0.1,T=4 : x@v(p) = 1.929, 3.003, 4.069   ratios = 1.000, 1.557, 2.109   (identity module {2,3,4}: ratios 1, 1.5, 2)
      per-rung implied v = 2.70, 2.60, 2.56

   (measured spatial v = 2.60)
      the same tower read with v=2: 1.483, 2.309, 3.129  — only the first value is near a rung
(iii)fixed-BC p=0, T=7 : x@v(0) = 1.953, 2.952, 3.934   ratios = 1.000, 1.512, 2.015   (the χ₁ template — same structure as (ii))


## 3. The unification: the partners ARE the missing states

If the visible tower is the *even* ladder, where did the *odd* states ($x = 1/2, 3/2, 5/2$ — the boundary-Majorana ladder) go? Answer: **to phase $\pi$** (NB7 Check 7). Reading every "partner" eigenvalue from the $\pi$ side,

$$x_{\rm partner} \;=\; \frac{v(p)\,T\,\big(\pi - |\Delta\phi|\big)}{\pi},$$

reproduces the *complete* odd ladder $\{1/2, 3/2, 5/2, 7/2\}$ — at every probed coupling ($p = 0.01$ to $0.15$) and every $T$ from 2 to 8, converging toward exact half-integers with $T$ exactly like ordinary rungs. Meanwhile every free-fermion control (Ising, and the XY family at all three velocities) has **zero** partner states, with its odd rungs sitting normally near phase 0.

So: at any $p\neq0$ the entire odd-fermion sector's eigenvalues acquire a factor $\approx e^{i\pi}$ — a **momentum-$\pi$ displacement of the odd tower**. Nothing was ever suppressed; a $|\Delta\phi|<\pi/2$ filter in our tower bookkeeping had been discarding the displaced states, and the "$\pm$ pairing" was the same tower seen from the other side. Its famous "monotone sharpening with $T$" is simply the $\tfrac12$-rung's offset $\pi x/(vT)\to0$. The cell below regenerates the central table.

In [3]:
# The unification table (caches: nb8_master, nb3_p01_stress, nb7_oddscan, nb7_xy_control).
master  = load("results/data/nb8_master.jld2", "done")
oddscan = load("results/data/nb7_oddscan.jld2", "done")

partner_x(thetas, lam0, T, v) = sort([v*T*(pi-abs(dphw(t,lam0)))/pi
                                      for t in thetas if t != lam0 && abs(dphw(t,lam0)) > pi/2])

println("p=0.1: partner states read as x = v(p)·T·(π−|Δφ|)/π   (odd-ladder prediction: 0.5, 1.5, 2.5, 3.5)")
for T in 2.0:2.0:8.0
    e = master[(0.1, T)]
    xs = partner_x(e.theta, e.theta_phys, T, v_p01)
    @printf("  T=%.0f (k=4): %s\n", T, join([@sprintf("%.3f", x) for x in xs], ", "))
end
th4 = sort(collect(deep[(0.1, 4.0, 8)].theta), by=abs, rev=true)
@printf("  T=4 (k=8): %s\n", join([@sprintf("%.3f", x) for x in partner_x(th4, th4[1], 4.0, v_p01)], ", "))

println("\ntiny-p probes (T=4): the displaced odd ladder exists at EVERY p≠0")
v_interp(p) = p <= 0.1 ? 1.977 + (2.601-1.977)*p/0.1 : 2.601 + (3.797-2.601)*(p-0.1)/0.2
for p in [0.01, 0.02, 0.05, 0.15]
    th = sort(collect(oddscan[p].theta), by=abs, rev=true)
    @printf("  p=%-5.2f : %s\n", p, join([@sprintf("%.3f", x) for x in partner_x(th, th[1], 4.0, v_interp(p))], ", "))
end

println("\nfree-fermion controls: partner count (prediction: zero)")
for T in (2.0, 8.0)
    e = master[(0.0, T)]
    @printf("  Ising p=0, T=%.0f : %d\n", T, length(partner_x(e.theta, e.theta_phys, T, 2.0)))
end
for g in (0.5, 1.0, 1.5)
    e = xy[(g, 6.0)]
    @printf("  XY γ=%.1f,  T=6 : %d\n", g, length(partner_x(e.theta, e.theta_phys, 6.0, 2g)))
end

p=0.1: partner states read as x = v(p)·T·(π−|Δφ|)/π   (odd-ladder prediction: 0.5, 1.5, 2.5, 3.5)


  T=2 (k=4): 0.456, 1.409
  T=4 (k=4): 0.468, 1.441


  T=6 (k=4): 0.469, 1.424
  T=8 (k=4): 0.470, 1.419
  T=4 (k=8): 0.468, 1.441, 2.504, 4.492

tiny-p probes (T=4): the displaced odd ladder exists at EVERY p≠0


  p=0.01  : 0.480, 1.457, 2.490, 3.578
  p=0.02  : 0.480, 1.459, 2.499, 3.637
  p=0.05  : 0.478, 1.442, 2.589, 4.116
  p=0.15  : 0.451, 1.432, 2.470, 3.333

free-fermion controls: partner count (prediction: zero)
  Ising p=0, T=2 : 0


  Ising p=0, T=8 : 0
  XY γ=0.5,  T=6 : 0
  XY γ=1.0,  T=6 : 0
  XY γ=1.5,  T=6 : 0


## Interlude — the fermion toolkit (everything §4–6 needs, in plain terms)

Sections 4–6 explain the *mechanism*, and they speak the language of fermions. Here is the minimum vocabulary, built from scratch; the CFT side (central charge, scaling dimensions, towers, boundary states) is assumed at the level of the companion `cft_primer.ipynb`.

**Spins → fermions (the Jordan–Wigner transformation).** A chain of quantum spins (each site up/down) can be rewritten *exactly* as a chain of fermions (each site empty/occupied) — no approximation, just a change of variables. The one subtlety: a fermion at site $j$ comes attached to a "string" of operators running back to the start of the chain (this enforces the minus-sign rule that makes fermions fermions). For the plain Ising chain this rewrite turns the Hamiltonian into *free* (non-interacting) fermions — which is exactly why Ising is exactly solvable. **Any term beyond nearest-neighbour — in particular Alcaraz's NNN coupling — turns into a product of *four* fermion operators: an *interaction*.** That single fact is the origin of everything in §4–6.

**Majoranas (half-fermions).** Each ordinary fermion can be split into two "Majorana" pieces $\gamma$, the way a complex number splits into real and imaginary parts. Concretely, per site $j$:

$$\gamma_{2j-1} = c_j + c_j^\dagger,\qquad \gamma_{2j} = i\,(c_j - c_j^\dagger),$$

so that each $\gamma$ is its *own* antiparticle ($\gamma^\dagger=\gamma$), squares to one ($\gamma_a^2=1$), and different ones anticommute: $\{\gamma_a,\gamma_b\}=2\delta_{ab}$. So $N$ spins $\to$ $2N$ Majoranas. They are the most convenient bookkeeping for this problem because the two terms of the Ising Hamiltonian become Majorana bonds on *alternating* links ($i\gamma_{2j}\gamma_{2j+1}$ and $i\gamma_{2j-1}\gamma_{2j}$), so the chain looks like one uniform Majorana chain at criticality — and "shift by one Majorana" is then a natural *half*-step of the lattice (used in §5).

**Fermion parity (even vs odd).** Count the fermions in a state; is that number even or odd? That even/odd label — *parity* — is conserved and is the hero of the story. In the transfer-matrix spectrum the two parities organize into two families:
- **even-parity states = the "identity module"** $\{0, 2, 3, 4, \ldots\}$ — the CFT calls these the *descendants of the identity* (states built by acting with the stress tensor; see `cft_primer.ipynb` §5). These are the $\{2,3,4\}$ ladder we read off at every $p$.
- **odd-parity states = the "$\varepsilon$ tower"** $\{1/2, 3/2, 5/2, \ldots\}$ — the boundary-Majorana ladder. These are the states that "go missing" at $p\neq0$ (and reappear, displaced by $\pi$, in §3).

So "the odd sector jumps to phase $\pi$" means, in plain terms: *the states with an odd number of fermions all get an extra minus sign under a sideways shift.* A minus sign is a phase of $\pi$ — that is the whole phenomenon.

**Kramers–Wannier duality (order ↔ disorder).** The Ising chain has a famous self-symmetry: swapping the two competing terms (the alignment term and the field term) leaves the critical point invariant — this is the duality that exchanges "ordered" and "disordered" descriptions. In Majorana language it is simply the *shift by one Majorana site*. Alcaraz's model was *designed* to keep this self-duality at $p\neq0$ (that is why its Hamiltonian carries both the NNN term and the XX term — §5 shows they are exact duals of each other). The relevance here: a *shift by half a spin site* is this duality operation, and doing it twice returns a full-site shift — the structure behind why the odd-sector phase is quantized at exactly $\pi$ rather than some tunable value ($\pi = $ the square of a half-step $\pm i$; §6/outlook).

With this vocabulary the mechanism reads in one line: *the NNN term is a four-fermion interaction; interactions can attach a minus sign to odd-parity states under translation; that minus sign is the $\pi$ displacement; and the model's Kramers–Wannier self-duality is what pins it to exactly $\pi$.*

### Interlude (derivation) — Jordan–Wigner step by step: where the 4-fermion interaction comes from

This section derives, with every step shown, how the Alcaraz Hamiltonian turns into fermions, and *why* two of its terms become **four-fermion interactions**. It is the algebra behind the one-line claim above ("anything beyond nearest-neighbour becomes a product of four fermion operators").

#### Why bother leaving spins at all?

Spins are algebraically awkward. Two Pauli operators on the **same** site *anti*commute ($\sigma^x\sigma^z=-\sigma^z\sigma^x$), but on **different** sites they *commute* ($[\sigma^a_i,\sigma^b_j]=0$ for $i\neq j$). So a spin is neither a boson (commutes everywhere) nor a fermion (anticommutes everywhere) — it is a "hard-core" object with a mixed algebra. There is no mode expansion, no Wick theorem, no notion of "free particles."

Fermions are the opposite: they anticommute *everywhere*, $\{c_i,c_j^\dagger\}=\delta_{ij}$, $\{c_i,c_j\}=0$. The payoff is enormous: **a Hamiltonian that is *quadratic* in $c,c^\dagger$ is exactly solvable** — you diagonalize a matrix, get independent normal modes $\eta_m$, and everything (spectrum, correlators, the transfer matrix) follows from single-particle data. That is the whole reason the transverse-field Ising chain is exactly solvable, and the deep reason its critical point is a *free* CFT ($c=1/2$, a single Majorana fermion).

The Jordan–Wigner (JW) transformation is an **exact** rewriting of the spin chain as a fermion chain — no approximation, just a change of variables. We do it because it sorts the Hamiltonian into two piles: the *quadratic* (free, solvable) part, and any *quartic* (interacting) leftovers. For the plain Ising chain the leftover pile is empty. For Alcaraz it is not — and the leftovers are exactly what create the $\pm$ partners.

#### The transformation (with the string, and why it is needed)

We want new operators that anticommute on different sites, built from spins that *commute* on different sites. JW achieves this by attaching a non-local **string** to each site. In the convention matched to our model (field on $\sigma^x$, order parameter on $\sigma^z$):

$$
\boxed{\;\sigma^x_j = 1 - 2n_j,\qquad
\sigma^z_j = \big(c_j + c_j^\dagger\big)\,K_j,\qquad
K_j \equiv \prod_{k<j}\big(1-2n_k\big)=\prod_{k<j}\big(-\sigma^x_k\big)\;}
$$

with $n_j=c_j^\dagger c_j$. The string $K_j=(-1)^{\#\text{fermions left of }j}$ is the crucial ingredient: without it, $(c_j+c_j^\dagger)$ on different sites would *commute* (they would behave bosonically). The string inserts exactly the minus signs that turn that commuting algebra into the anticommuting fermion algebra $\{c_i,c_j^\dagger\}=\delta_{ij}$. It also means $\sigma^z$ is *non-local* in the fermions — and that non-locality is what makes long-range spin terms become multi-fermion.

#### Two identities we will use repeatedly

Because $n$ is a projector ($n^2=n$), one checks directly (expand and use $cc=0,\ c^\dagger c^\dagger=0$):

$$
(c+c^\dagger)(1-2n) = c^\dagger - c,\qquad\qquad K_j^2 = 1 .
$$

*(Check of the first: $(c)(1-2n)=c-2cc^\dagger c=c-2c=-c$, using $cc^\dagger c=c$; and $(c^\dagger)(1-2n)=c^\dagger-2c^\dagger c^\dagger c=c^\dagger$. Sum $=c^\dagger-c$.)*

#### Term by term

Recall $H=-\sum_i\big[\ \sigma^z_i\sigma^z_{i+1} \;+\; p\,\sigma^z_i\sigma^z_{i+2} \;+\; \lambda\,\sigma^x_i \;+\; p\lambda\,\sigma^x_i\sigma^x_{i+1}\ \big]$.

**(1) Field $\lambda\,\sigma^x_j$.** Directly,
$$\lambda\,\sigma^x_j = \lambda\,(1-2n_j).$$
**Quadratic** (a chemical potential). Free. ✔

**(2) Nearest-neighbour $\sigma^z_j\sigma^z_{j+1}$ (the Ising bond).** Using $K_{j+1}=K_j(1-2n_j)$ and that $(c_{j+1}+c_{j+1}^\dagger)$ commutes with $K_j$ (which only touches sites $<j$), and $K_j^2=1$:
$$
\sigma^z_j\sigma^z_{j+1}
=(c_j+c_j^\dagger)K_j\,(c_{j+1}+c_{j+1}^\dagger)K_{j+1}
=(c_j+c_j^\dagger)(1-2n_j)\,(c_{j+1}+c_{j+1}^\dagger)
=(c_j^\dagger-c_j)(c_{j+1}+c_{j+1}^\dagger).
$$
The string **telescoped away** — the two adjacent strings overlap on all sites $<j$ and square to $1$, leaving nothing between $j$ and $j+1$. The result is **quadratic**: hopping $c_j^\dagger c_{j+1}$ + pairing $c_j^\dagger c_{j+1}^\dagger$ + h.c. Free. ✔ This is why Ising ($=$ terms (1)+(2)) is a free-fermion theory.

**(3) The $\;p\lambda\,\sigma^x_j\sigma^x_{j+1}$ term.**
$$
p\lambda\,\sigma^x_j\sigma^x_{j+1}=p\lambda\,(1-2n_j)(1-2n_{j+1}).
$$
Expanding, $(1-2n_j)(1-2n_{j+1})=1-2n_j-2n_{j+1}+4\,n_jn_{j+1}$, and $n_jn_{j+1}=c_j^\dagger c_j\,c_{j+1}^\dagger c_{j+1}$ is a product of **four** fermion operators. This is a **quartic density–density interaction** ✘ — it is diagonal in the occupation basis, so it does **not** move fermions; it only assigns an energy to *adjacent occupied* sites.

**(4) The next-nearest-neighbour $\;p\,\sigma^z_i\sigma^z_{i+2}$ term.** Now the strings do *not* fully telescope. With $K_{i+2}=K_i(1-2n_i)(1-2n_{i+1})$:
$$
\sigma^z_i\sigma^z_{i+2}
=(c_i+c_i^\dagger)K_i\,(c_{i+2}+c_{i+2}^\dagger)K_{i+2}
=(c_i+c_i^\dagger)(1-2n_i)\,(1-2n_{i+1})\,(c_{i+2}+c_{i+2}^\dagger)
=(c_i^\dagger-c_i)\,\underbrace{(1-2n_{i+1})}_{=\,(-1)^{n_{i+1}}}\,(c_{i+2}+c_{i+2}^\dagger).
$$
A leftover factor $(1-2n_{i+1})$ survives — the parity of the site **skipped over**. This is again **quartic** ✘, but of a completely different character from (3): it is a **next-nearest hop** (it *does* move a fermion, from $i$ to $i+2$) **dressed by the parity** $(-1)^{n_{i+1}}$ of the intermediate site — a *parity-dressed hopping*.

#### The punchline

$$
\begin{array}{c|c|c}
\textbf{spin term} & \textbf{fermion form} & \textbf{type}\\\hline
\lambda\,\sigma^x & \lambda(1-2n) & \text{quadratic (free)}\\
\sigma^z_i\sigma^z_{i+1} & (c_i^\dagger-c_i)(c_{i+1}+c_{i+1}^\dagger) & \text{quadratic (free)}\\
p\lambda\,\sigma^x_i\sigma^x_{i+1} & p\lambda(1-2n_i)(1-2n_{i+1}) & \text{quartic: density–density}\\
p\,\sigma^z_i\sigma^z_{i+2} & p\,(c_i^\dagger-c_i)(1-2n_{i+1})(c_{i+2}+c_{i+2}^\dagger) & \text{quartic: parity-dressed hop-2}
\end{array}
$$

So Ising = quadratic = free fermions = exactly solvable = a $c=1/2$ Majorana CFT. The two Alcaraz additions are **both** four-fermion interactions, and they are physically distinct: the $XX$ term is a *density–density* interaction (diagonal, moves nothing), while the NNN-$ZZ$ term is a *parity-dressed hop* (moves fermions by two, with a sign set by the occupation of the site in between). This is exactly the distinction the term-dissection probe (§5, NB7 Check 8) tests — and it finds that only the **parity-dressed hop** installs the momentum-$\pi$ displacement that creates the odd-tower partners; the density–density term does not. Quartic terms are what break integrability: fermions now *scatter* off each other, Wick's theorem fails, and (as §4 shows) the transfer matrix is no longer a free-fermion Gaussian — the very condition that let §4 prove a free chain *cannot* build the $\pi$-displaced partners at full modulus.

### Interlude (theory) — Gaussian operators, Wick's theorem, and what quartic terms destroy

The derivation above sorted the Hamiltonian into *quadratic* and *quartic* piles. This section explains **why that split is the whole ballgame**: what "free/Gaussian" buys you (Wick's theorem, a normal form for the transfer matrix), and precisely what breaks when quartic terms appear. Everything §4 argues rests on this.

#### Gaussian (a.k.a. "free") operators and states

An operator is **Gaussian** if it is the exponential of an expression that is *quadratic* in the fermions:

$$ \mathcal{O} \;=\; \exp\Big(\textstyle\sum_{ij} A_{ij}\,c_i^\dagger c_j \;+\; \tfrac12\sum_{ij}\big(B_{ij}\,c^\dagger_i c^\dagger_j + \text{h.c.}\big)\Big).$$

The name comes from the analogy with Gaussian probability distributions: just as a Gaussian distribution is fully specified by its mean and *covariance* (its two-point data), a Gaussian fermion state is fully specified by the matrix of two-point functions $\langle c^\dagger_i c_j\rangle$, $\langle c_ic_j\rangle$ — **the covariance matrix**. Nothing else is independent information.

Why the terms with $c^\dagger c^\dagger$ (pairing) matter: with them present you cannot diagonalize by simply relabelling particles — you must mix creation and annihilation operators,
$$\eta_m \;=\; \sum_j\big(u_{mj}\,c_j + v_{mj}\,c_j^\dagger\big),$$
which is called a **Bogoliubov transformation**. It is still a *linear* change of variables, so the result is still free: the Hamiltonian becomes $H=\sum_m \varepsilon_m\,\eta^\dagger_m\eta_m + \text{const}$, a set of **independent normal modes** with single-particle energies $\varepsilon_m$. (Our Ising bond, term (2) above, produced exactly such a pairing term — hence Bogoliubov is needed even for plain Ising.)

#### Wick's theorem

**Statement.** In a Gaussian state, *every* higher correlation function collapses into sums of products of **two-point** functions ("contractions"), one term per way of pairing the operators, with a sign given by the parity of the permutation that brings the paired partners together. For four operators:

$$\langle c_1c_2c_3c_4\rangle \;=\; \langle c_1c_2\rangle\langle c_3c_4\rangle \;-\; \langle c_1c_3\rangle\langle c_2c_4\rangle \;+\; \langle c_1c_4\rangle\langle c_2c_3\rangle .$$

(The alternating signs are the fermionic bookkeeping: exchanging two fermion operators costs a minus.)

**Why it holds.** A Gaussian weight $e^{\text{quadratic}}$ makes the many-body problem *linear* in the operators: the modes $\eta_m$ are independent, each mode is a two-level system with its own occupation, and expectation values factorize mode by mode. Nothing correlates three or four particles beyond what pairs already encode.

**Why it is such a big deal.** It means the entire theory — all correlators, the entanglement structure, the full spectrum — is generated by *single-particle* data (an $N\times N$ or $2N\times2N$ matrix), not by an exponentially large Hilbert space. This is exactly the sense in which free = "exactly solvable", and why the Ising critical point is a *free* CFT built from one Majorana field.

#### The consequence for our transfer matrix: a normal form

If $E$ is Gaussian, the same Bogoliubov logic puts it in the **normal form** used in §4:

$$ E \;=\; \lambda_0\,\exp\Big(-\sum_m \varepsilon_m\,\hat n_m\Big),\qquad \hat n_m=\eta^\dagger_m\eta_m\in\{0,1\}. $$

Its eigenvalues are then completely explicit — pick any subset $S$ of modes to occupy:

$$ \lambda_S=\lambda_0\,e^{-\sum_{m\in S}\varepsilon_m}, \qquad
|\lambda_S| = |\lambda_0|\,e^{-\sum_{m\in S}\mathrm{Re}\,\varepsilon_m}, \qquad
\arg\lambda_S = \arg\lambda_0-\sum_{m\in S}\mathrm{Im}\,\varepsilon_m .$$

Read that last pair carefully, because it is the engine of §4: **a free spectrum's moduli and phases are both *additive over occupied modes*.** $\mathrm{Re}\,\varepsilon_m$ is what a mode costs you in magnitude; $\mathrm{Im}\,\varepsilon_m$ is the phase it contributes. This is also *why* the CFT tower appears as small, evenly-spaced phase steps: the near-gapless modes have $\mathrm{Re}\,\varepsilon\approx0$ and $\mathrm{Im}\,\varepsilon=\pi x/(vT)$.

#### What quartic terms destroy

A quartic term is **not** a quadratic form, so $e^{H}$ with a quartic $H$ is **not Gaussian**. Three things fail at once:

1. **Wick's theorem fails.** $\langle c_1c_2c_3c_4\rangle$ is no longer just the three pairings: it acquires a genuinely *connected* (irreducible) four-point piece that no product of two-point functions reproduces. The covariance matrix stops being a complete description.
2. **Modes stop being independent.** With $n_in_j$ or a parity-dressed hop present, occupying one mode changes the cost of another — i.e. the particles **scatter** off one another. There is no longer a set of conserved single-mode occupations, hence no normal form, hence no additive-phase rule above.
3. **Integrability is generically lost.** "Integrable" means the model retains an extensive set of conserved quantities that make scattering trivially factorized (particles pass through each other keeping their momenta). Free fermions are the extreme case. A generic quartic term destroys those conservation laws, and the model becomes non-integrable — which is precisely the class Alcaraz's model sits in, and the reason it has no exact solution to compare against.

#### The sharpened free-fermion obstruction (a correction worth stating)

§4 argues that a *free* chain cannot produce the $\pi$-displaced tower. The **load-bearing** part of that argument is the additivity above and is correct: to sit at nearly full modulus, an eigenvalue may only occupy modes with $\mathrm{Re}\,\varepsilon_m\approx0$; in an Ising-class critical chain those are the long-wavelength modes, whose phases $\pi x/(vT)$ are *small*. Accumulating exactly $\pi$ would need a mode that is simultaneously **non-decaying** ($\mathrm{Re}\,\varepsilon=0$) and **maximally oscillating** ($\mathrm{Im}\,\varepsilon=\pi$) — a gapless branch at lattice momentum $\pi$ — which the uniform Ising-class chain does not have.

One step as originally phrased, however, needs sharpening. It said the required object — a phase counting fermion parity — is "an all-orders, maximally non-Gaussian operator, so only interactions can write it." That is **not** quite right: the parity *operator* is
$$(-1)^{\hat N}=\prod_j\big(1-2n_j\big)=e^{\,i\pi \hat N},\qquad \hat N=\sum_j c^\dagger_jc_j,$$
and $\hat N$ is **quadratic** — so $e^{i\pi\hat N}$ is itself a perfectly Gaussian operator. (Multiplying every $\varepsilon_m$ by an extra $+i\pi$ would give a *free* transfer matrix whose odd-parity eigenvalues carry $e^{i\pi}$ at unchanged modulus.) What is genuinely non-Gaussian is the parity *projector* $\hat P_{\rm odd}=\tfrac12\big(1-(-1)^{\hat N}\big)$ — but the displacement only needs the operator, not the projector.

So the correct necessity statement is **not** "parity is non-Gaussian, hence interactions", but the dispersion statement: *the uniform, untwisted Ising-class free chain has no zero-decay momentum-$\pi$ mode, so its near-top eigenvalues cannot carry phase $\pi$* — confirmed by the zero-partner column for Ising and for XY at all three velocities. The displacement therefore requires *some* source of momentum-$\pi$ structure; it does not have to be a bulk interaction. §7 is exactly that lesson: **XXZ gets its $\pi$ from a staggered (period-2) boundary**, not from a quartic bulk term, while **Alcaraz gets it from the bulk parity-dressed hop** (§5). Two different switches, one spectral fingerprint — which is why §5's term-dissection (not a generic "interactions do it" argument) is what actually identifies the driver here.

### Interlude (glossary) — the remaining terms used later, in plain language

Everything above covers the fermion side. A few more terms appear in §5–§8; here is each one, briefly, so nothing in this notebook is used without a definition somewhere in it.

**Stress tensor, primaries, descendants, "modules".** In a CFT the operators come in families. Each family has one *primary* operator (the "lowest rung") plus infinitely many *descendants*, obtained by acting with the stress tensor $T$ — the operator that generates translations/rescalings, i.e. the CFT's energy–momentum. A family (primary + all its descendants) is a *module*; its list of scaling dimensions is packaged by a **character** $\chi$. Critical Ising has exactly three families:

$$\chi_{1}:\;\{0,2,3,4,\ldots\}\quad(\text{identity}),\qquad
\chi_{\varepsilon}:\;\{\tfrac12,\tfrac32,\tfrac52,\ldots\}\quad(\text{energy}),\qquad
\chi_{\sigma}:\;\{\tfrac1{16},\tfrac{17}{16},\ldots\}\quad(\text{spin}).$$

Which families appear depends on the **boundary condition**: our free boundary $|X^+\rangle$ realizes $\chi_1\oplus\chi_\varepsilon$ (hence the interleaved $\{0,\tfrac12,\tfrac32,2,\ldots\}$ ladder, and $x_1=1/2$), while a fixed boundary realizes $\chi_1$ alone (hence $x_1=2$). In this notebook's language: $\chi_1$ = the even-parity tower, $\chi_\varepsilon$ = the odd-parity tower that gets displaced by $\pi$. (Built from scratch in `cft_primer.ipynb`.)

**Conformal width $W=vT$.** The transfer-matrix phases are $\pi x/(vT)$, so it is the *product* $vT$ — not $T$ alone — that sets how compressed the tower is. $W=vT$ is the physical width of the conformal strip. Two models are only being compared like-for-like at equal $W$: this is why §6 quotes Ising as clean out to $W\gtrsim36$ against Alcaraz walling at $W\approx26$, rather than comparing raw $T$.

**Oblique Rayleigh–Ritz (Petrov–Galerkin) — what the block solver actually does.** To get the top few eigenvalues of a huge operator $E$ you keep a small $k$-dimensional subspace and ask for the best approximate eigenpairs *inside* it. Because $E$ is non-Hermitian, its left and right eigenvectors differ, so one keeps **two** bases — $k$ right states $\{R_j\}$ and $k$ left states $\{L_i\}$ — and projects with the left basis onto the right one. That asymmetry is what "oblique" means (an ordinary Rayleigh–Ritz uses one orthonormal basis). It produces two small $k\times k$ matrices, the overlap $S_{ij}=\langle L_i|R_j\rangle$ and $M_{ij}=\langle L_i|E|R_j\rangle$ — together called a **matrix pencil** $(M,S)$ — whose eigenvalues (the **Ritz values**) approximate the true ones. Crucially the pencil sorts states by *complex value*, not by modulus, which is exactly why it separates the two towers that modulus-ranking confuses (§6b).

**Eigenvalue vs eigenvector conditioning; exceptional points.** "Conditioning" measures how much an answer moves when the input is perturbed slightly. For a non-Hermitian operator these two are very different: the eigen*values* stay well-conditioned even near a degeneracy, while the eigen*vector* sensitivity grows like $1/(\text{gap})$ — so as two eigenvalues approach, the *direction* of the eigenvector becomes numerically meaningless long before its value does. The extreme case is an **exceptional point**, where two eigenvectors do not merely get close but *coalesce* into one (the operator stops being diagonalizable). This is the distinction behind §6: the eigenvalue routes survive the wall, the entropy (which needs the eigen*vector*) does not. Quantified in NB5 and NB8.

**Self-duality and the half-step.** A model is *self-dual* if a transformation maps it back to itself with couplings exchanged — for Ising, Kramers–Wannier swaps the field and the bond term, and the critical point is the fixed point of that swap. On the Majorana chain this transformation is literally a **shift by one Majorana**, i.e. *half* a spin site. Since two half-steps make one full site translation, $E=E_{1/2}^2$, and any $\pm i$ carried by a sector's half-step squares to exactly $-1=e^{i\pi}$ — which is the structural reason the displacement is pinned at $\pi$ rather than taking a coupling-dependent value (§5).

## 4. Why free fermions cannot do this (a real derivation)

For any free-fermion model the transfer matrix has a normal form $E = \lambda_0\,\exp(-\sum_m \varepsilon_m \hat n_m)$: a set of *independent modes*, each contributing a complex cost $\varepsilon_m$ when occupied — a decay part ($\mathrm{Re}$) and a phase part ($\mathrm{Im}$). An eigenvalue near the top ($|\lambda|\approx|\lambda_0|$) can only occupy modes with $\mathrm{Re}\,\varepsilon_m\approx0$, and those are precisely the long-wavelength CFT modes with tiny phases $\pi x/(vT)$. **Phases can only be accumulated in small increments**: assembling a lump of exactly $\pi$ at full modulus would require a mode that oscillates maximally *and* decays not at all — a gapless branch at lattice momentum $\pi$ — which no Ising-class free chain possesses. Hence *free $\Rightarrow$ no displaced tower*. The numerical support is the zero-partner column of the table above: four independent free-fermion datasets (Ising, and XY at $v=1,2,3$), zero partners in every one.

Conversely, the displaced tower means $\log E$ contains a term that *counts fermion parity*, $\approx i\pi\hat N$ — every odd-parity state picking up $e^{i\pi}=-1$. **Caution (see the Wick/Gaussian interlude above):** it is tempting to call this "maximally non-Gaussian, so only interactions can write it", but that is not right — $\hat N$ is quadratic, so $e^{i\pi\hat N}$ is itself Gaussian. The necessity direction is the *dispersion* argument of the previous paragraph: the uniform Ising-class free chain simply has no zero-decay momentum-$\pi$ branch to supply that phase, which is what the zero-partner controls confirm. The displacement therefore needs *some* source of momentum-$\pi$ structure — a bulk parity-dressed term (Alcaraz, §5) or a staggered boundary (XXZ, §7). Which one operates here is the next question.

## 5. Which term installs the π — and the duality structure behind it

The Alcaraz deformation switches on *two* terms at once, and both are quartic after Jordan–Wigner — but they are structurally different (NB7 Check 8):

| term | fermion language | π-partners? (probe below) |
|---|---|---|
| $-g\,X_iX_{i+1}$ | $(1-2n_i)(1-2n_{i+1})$ — plain **density–density** | **no** |
| $-g\,Z_iZ_{i+2}$ | $(c^\dagger_i-c_i)\,(1-2n_{i+1})\,(c^\dagger_{i+2}+c_{i+2})$ — **hopping dressed by the skipped site's parity** | **yes** |

So the driver is not "interactions" generically: it is **parity-dressed hopping** — a fermion that hops *over* a site picks up $(-1)$ if that site is occupied.

**The duality structure (and why the phase is exactly $\pi$).** Write the chain in Majorana variables $\gamma_1, \gamma_2, \ldots$ (two per spin site). The TFIM terms are the bilinears on alternating bonds of the Majorana chain, and the two quartic terms are *four consecutive Majoranas*, differing only by a one-site offset:

$$X_jX_{j+1} \propto \gamma_{2j-1}\gamma_{2j}\gamma_{2j+1}\gamma_{2j+2}, \qquad Z_jZ_{j+2} \propto \gamma_{2j}\gamma_{2j+1}\gamma_{2j+2}\gamma_{2j+3}.$$

The shift by one Majorana site — which **is** the lattice Kramers–Wannier duality (it swaps $X_j \leftrightarrow Z_jZ_{j+1}$, order $\leftrightarrow$ disorder) — maps the XX term exactly onto the NNN-ZZ term. The two probed deformations are KW duals of each other, and their $\lambda$-weighted sum is KW-invariant: *this is precisely why the Alcaraz Hamiltonian needs both terms to stay self-dual* (CLAUDE.md §2), now visible as elementary algebra. The cell below verifies every one of these operator identities numerically on a small chain, to machine precision.

The quantization then has a natural home: at the self-dual point the network admits a *half-step* transfer operation $E_{1/2}$ (translate by one Majorana site), with $E = E_{1/2}^2$; a sector whose half-step eigenvalue carries $\pm i$ lands, after squaring, at phase **exactly $\pi$** — a discrete index, not a coupling-dependent phase (consistent with the offset being identical at $p=0.01$ and $p=0.15$ in the table of §3). **Still open**: the first-principles selection of which sector carries the $\pm i$, and why the non-self-dual NNN-only variant retains the displacement while XX-only does not (the discreteness means weak duality-breaking cannot remove it continuously, but the selection itself needs a derivation — stated precisely in `next_steps.md`).

In [4]:
# Numerical verification of the Majorana/Kramers–Wannier algebra (dense, N=5 spins, machine precision).
const I2 = [1.0 0; 0 1]; const Xm = [0.0 1; 1 0]; const Ym = [0 -1.0im; 1.0im 0]; const Zm = [1.0 0; 0 -1]
N = 5
site_op(O, j) = kron([l == j ? O : I2 for l in 1:N]...)
gamma(m) = begin                                    # JW Majoranas, field-on-X convention
    j = (m + 1) ÷ 2
    string_part = j == 1 ? Matrix{ComplexF64}(I, 2^N, 2^N) : prod(site_op(Xm, l) for l in 1:j-1)
    string_part * site_op(isodd(m) ? Zm : Ym, j)
end

algebra_ok = all(isapprox(gamma(m)*gamma(n) + gamma(n)*gamma(m), (m == n ? 2.0 : 0.0)*I, atol=1e-10)
                 for m in 1:2N for n in m:2N)
println("Majorana algebra {γ_m, γ_n} = 2δ_mn : ", algebra_ok)

# proportionality helper: returns the constant c with A = c·B (checks it is exact)
prop(A, B) = begin
    c = tr(B' * A) / tr(B' * B)
    isapprox(A, c * B, atol=1e-10) ? c : NaN
end
j = 2
println("X_j          = (", prop(site_op(Xm,j), gamma(2j-1)*gamma(2j)), ")·γ_{2j-1}γ_{2j}      [TFIM field term]")
println("Z_jZ_{j+1}   = (", prop(site_op(Zm,j)*site_op(Zm,j+1), gamma(2j)*gamma(2j+1)), ")·γ_{2j}γ_{2j+1}      [TFIM bond term]")
println("X_jX_{j+1}   = (", prop(site_op(Xm,j)*site_op(Xm,j+1),
        gamma(2j-1)*gamma(2j)*gamma(2j+1)*gamma(2j+2)), ")·γ_{2j-1}γ_{2j}γ_{2j+1}γ_{2j+2}")
println("Z_jZ_{j+2}   = (", prop(site_op(Zm,j)*site_op(Zm,j+2),
        gamma(2j)*gamma(2j+1)*gamma(2j+2)*gamma(2j+3)), ")·γ_{2j}γ_{2j+1}γ_{2j+2}γ_{2j+3}")
println("→ both quartic terms are 4-consecutive-Majorana products, offset by ONE Majorana site:")
println("  the γ-index shift m→m+1 (= lattice Kramers–Wannier) maps X_jX_{j+1} onto Z_jZ_{j+2}. ∎")

Majorana algebra {γ_m, γ_n} = 2δ_mn : true


X_j          = (

0.0 + 1.0im)·γ_{2j-1}γ_{2j}      [TFIM field term]
Z_jZ_{j+1}   = (0.0 + 1.0im)·γ_{2j}γ_{2j+1}      [TFIM bond term]
X_jX_{j+1}   = (-1.0 + 0.0im)·γ_{2j-1}γ_{2j}γ_{2j+1}γ_{2j+2}
Z_jZ_{j+2}   = (-1.0 + 0.0im)·γ_{2j}γ_{2j+1}γ_{2j+2}γ_{2j+3}
→ both quartic terms are 4-consecutive-Majorana products, offset by ONE Majorana site:
  the γ-index shift m→m+1 (= lattice Kramers–Wannier) maps X_jX_{j+1} onto Z_jZ_{j+2}. ∎


In [5]:
# The term-dissection probe (cache: nb7_variants.jld2, NB7 Check 8).
variants = load("results/data/nb7_variants.jld2", "done")
println("Single-term deformations of critical Ising (g=0.1, T=4, k=6; x read with v=2):")
for key in [(:nnn, 0.1), (:xx, 0.1)]
    e = variants[key]
    ths = sort(collect(e.theta), by=abs, rev=true)
    lam0 = ths[1]
    tower    = sort([abs(dphw(t, lam0)) for t in ths[2:end] if abs(dphw(t, lam0)) < pi/2])
    partners = sort([abs(dphw(t, lam0)) for t in ths[2:end] if abs(dphw(t, lam0)) > pi/2], rev=true)
    @printf("  %-5s tower x: %-34s partners x: %s\n", string(key[1]),
            join([@sprintf("%.3f", 2*4*g/pi) for g in tower], ", "),
            isempty(partners) ? "(none)" : join([@sprintf("%.3f", 2*4*(pi-g)/pi) for g in partners], ", "))
end

Single-term deformations of critical Ising (g=0.1, T=4, k=6; x read with v=2):


  nnn   tower x: 0.830, 3.229                       partners x: 0.371, 1.216, 2.170
  xx    tower x: 0.393, 1.242, 1.647, 2.185, 3.177  partners x: (none)


## 6. The methods payoff: what exactly breaks, and why, route by route

The displaced tower *reframes* the old "frustration closes the gap faster" story (CLAUDE.md §17). The precise statement needs one distinction: **different numerical tasks care about different distances in the spectrum.** Power iteration and modulus-ranking care about $|\lambda|$ *magnitude* gaps; eigenvector conditioning cares about distances in the *complex plane*. The two towers are modulus-degenerate but sit $\pi$ apart in phase — close in magnitude, far in value — and that is exactly why different routes break differently:

**(a) Naive routes — broken entirely by the partner tower.** A single-vector power method converges to the largest-*modulus* eigenvalue; a modulus-ranking selector picks by $|\lambda|$. The partner is modulus-degenerate with $\lambda_0$ from the start ($0.94$ at $T=2$) and **crosses above $|\lambda_0|$ at $T\gtrsim6$** (table below) — so these routes lock onto the wrong state or oscillate between the two towers. This is 100% the NNN-induced two-tower structure, and has *nothing* to do with any gap closing: it would happen even if the intra-tower gaps stayed wide open. It retroactively explains the "$-\lambda_0$ contamination" of the early entropy domes, why `pick_phys` complex-value continuity became load-bearing, and the cold-started cluster branch jumps at $T=12/13$.

**(b) Block eigenvalues — never broken.** The oblique Rayleigh–Ritz pencil separates states by complex *value*, where the towers are far apart. Validated to $10^{-8}$–$10^{-13}$ against exact diagonalization (NB5).

**(c) The eigenvector/entropy wall ($T\approx10$ at $p=0.1$).** What ill-conditions the physical eigenvector is near-degeneracy in complex *value* near $\lambda_0$ — the *intra-tower* ladder compressing as $\pi x/(vT)\to0$ — and here the comparison with Ising is sharp: **Ising shows no wall at all in the tested range** (our runs to $T=14$, the original paper's to $T=18$, i.e. clean out to conformal width $W = vT \gtrsim 36$), while Alcaraz $p=0.1$ walls already at $W \approx 2.6\times10 = 26$. Since the intra-tower gap at fixed $W$ is the same CFT quantity for both, the earlier wall is **not** explained by tower compression alone — the $p\neq0$ structure genuinely lowers the wall threshold. Two identified contributors: the larger $v(p)$ compresses the ladder at fixed $T$ (a factor $1.32$ at $p=0.1$), and the partner cluster consumes half the $k$-block's capacity while demanding value-aware selection — the doubled cluster makes the block's job strictly harder at the same $T$. Disentangling the two quantitatively is an open item; the sharp cluster test is whether $T_{\rm wall}(p)\cdot v(p)\approx26$ holds across the p-sweep (an Alcaraz-family law — with Ising, clean at $W\gtrsim36$, explicitly *not* on it, which would pin the difference on the partner structure rather than the velocity).

In [6]:
# The two competing near-degeneracies, p=0.1 vs p=0 (cache: nb8_master, the production k=4 sweep).
println("largest same-tower |λ/λ0|   vs   partner |λ/λ0|      (p=0 has no partners)")
@printf("  %-4s  %-12s %-12s | %-12s\n", "T", "p=.1 tower", "p=.1 partner", "p=0 tower")
for T in 2.0:1.0:9.0
    e1 = master[(0.1, T)]
    tow = [abs(e1.theta[j])/abs(e1.theta_phys) for j in eachindex(e1.theta)
           if j != e1.i0 && abs(dphw(e1.theta[j], e1.theta_phys)) < pi/2]
    par = [abs(e1.theta[j])/abs(e1.theta_phys) for j in eachindex(e1.theta)
           if j != e1.i0 && abs(dphw(e1.theta[j], e1.theta_phys)) > pi/2]
    e0 = master[(0.0, T)]
    tow0 = [abs(e0.theta[j])/abs(e0.theta_phys) for j in eachindex(e0.theta)
            if j != e0.i0 && abs(dphw(e0.theta[j], e0.theta_phys)) < pi/2]
    @printf("  %-4.0f  %-12.4f %-12.4f | %-12.4f\n", T,
            isempty(tow) ? NaN : maximum(tow), isempty(par) ? NaN : maximum(par),
            isempty(tow0) ? NaN : maximum(tow0))
end
println("\n  → the partner crosses |λ0| at T≳6: the modulus-dominant state is then the displaced")
println("    odd state, and any modulus-ranking selector picks the wrong branch.")

largest same-tower |λ/λ0|   vs   partner |λ/λ0|      (p=0 has no partners)
  T     p=.1 tower   p=.1 partner | p=0 tower   
  2     0.6510       0.9398       | 0.8897      


  3     0.8644       0.9783       | 0.9466      
  4     0.9352       0.9917       | 0.9687      
  5     0.9648       0.9979       | 0.9793      
  6     0.9799       1.0012       | 0.9852      
  7     0.9887       1.0032       | 0.9887      
  8     0.9942       1.0045       | 0.9911      
  9     0.9976       1.0050       | 0.9928      

  → the partner crosses |λ0| at T≳6: the modulus-dominant state is then the displaced
    odd state, and any modulus-ranking selector picks the wrong branch.


## 7. Corroboration from XXZ: the same fingerprint by a different switch

The XXZ Néel quench is worth a brief look purely as *supporting evidence* for the Alcaraz mechanism: it is the campaign's other interacting model, its wall was independently traced to a 4-fold "2 sectors × ± partners" cluster (NB9 §4/§4b, NB12 §4d), and the cell below shows — from the cached spectra — that its cluster carries the identical $\pi$-displacement fingerprint: a tight modulus band whose cross-pairs sit $\approx\pi$ apart in phase.

The instructive difference is *how* the $\pi$ arises. In XXZ it comes from the **boundary**: the Néel state is a checkerboard, breaking translation-by-one to translation-by-two, so its two sectors differ by lattice momentum $\pi$ by construction — no parity-dressed bulk term needed. In Alcaraz the boundary is uniform and the $\pi$ is generated by the **bulk** NNN term (§5). Seeing the same spectral structure emerge from two different microscopic switches is what confirms that the object itself — a momentum-$\pi$ displaced partner tower degenerate in modulus with the physical one — is the general mechanism behind the method's walls, with Alcaraz's route (parity-dressed hopping) being the one relevant to this thesis.

In [7]:
# XXZ's 4-fold cluster from cache (nb10_xxz_gap_k6.jld2, NB9 §4b): moduli + pairwise phase gaps.
xxz = load("results/data/nb10_xxz_gap_k6.jld2", "d6")
for T in (4.0, 5.0)
    th = sort(collect(xxz[T].theta), by=abs, rev=true)[1:4]
    @printf("T=%.0f  leading-4 |θ|: %s   (a %d%%-tight modulus band)\n", T,
            join([@sprintf("%.3f", abs(t)) for t in th], ", "),
            round(Int, 100*(1 - abs(th[4])/abs(th[1]))))
    println("     pairwise |Δφ| (rad):")
    for a in 1:4, b in a+1:4
        g = abs(dphw(th[a], th[b]))
        marker = abs(g - pi) < 0.35 ? "   ← ≈π (displaced cross-pair)" : ""
        @printf("       (%d,%d): %.2f%s\n", a, b, g, marker)
    end
end

T=4  leading-4 |θ|: 0.893, 0.885, 0.867, 0.867   (a 3%-tight modulus band)
     pairwise |Δφ| (rad):


       (1,2): 2.40


       (1,3): 2.67
       (1,4): 0.47
       (2,3): 0.27
       (2,4): 2.87   ← ≈π (displaced cross-pair)
       (3,4): 3.14   ← ≈π (displaced cross-pair)
T=5  leading-4 |θ|: 0.898, 0.892, 0.880, 0.880   (a 2%-tight modulus band)
     pairwise |Δφ| (rad):
       (1,2): 2.59
       (1,3): 0.19
       (1,4): 2.95   ← ≈π (displaced cross-pair)
       (2,3): 2.78
       (2,4): 0.36
       (3,4): 3.14   ← ≈π (displaced cross-pair)


## 8. Summary, and what remains

**Established — with its evidence location:**

| finding | numerical support |
|---|---|
| the pipeline is velocity-faithful | §2 cell (XY: $a_1$ vs $\pi/(4\gamma)$ at $v=1,2,3$); NB7 Check 4 |
| the Alcaraz $p=0.1$ tower = identity module at the TRUE $v(p)$ | §2 cell (rungs, ratios, per-rung implied $v$); NB7 Checks 2, 6 |
| the ± partners = Alcaraz's odd ladder displaced by exactly $\pi$, at every $p\neq0$ | §3 cell (the unification table); NB7 Check 7 |
| free fermions cannot displace (necessity of interactions) | derivation §4 + zero-partner controls in the §3 cell |
| the driver is Alcaraz's parity-dressed NNN hopping, not density–density | §5 probe cell; NB7 Check 8 |
| XX and NNN-ZZ are exact Kramers–Wannier duals (⇒ why the Alcaraz Hamiltonian's self-duality needs both; ⇒ the exact-$\pi$ quantization via the half-step structure) | §5 Majorana-algebra cell (machine-precision operator identities) |
| the Alcaraz block-PM wall is the doubled top cluster, not a faster gap; the partner is modulus-dominant at $T\gtrsim6$; the reach hierarchy tracks top-cluster multiplicity | §6 cell (the modulus table); NB5/NB8 for the wall itself |
| corroboration: XXZ shows the same $\pi$-fingerprint via its staggered boundary | §7 cell (4-fold band, $\approx\pi$ cross-pairs); NB9 §4/§4b |

**Open (all Alcaraz-facing):**
- The first-principles derivation selecting which deformations carry the displacement index (the well-posed $\mathbb{Z}_2$-background/parity-string problem, `next_steps.md`).
- A sub-1 tower state at $p\ge0.15$ ($x\approx0.83$–$0.94$, $T$-present, velocity excuse ruled out by direct measurement) that **matches no Ising boundary dimension** — the three boundary spectra $\chi_1=\{0,2,3,\ldots\}$, $\chi_\varepsilon=\{\tfrac12,\tfrac32,\ldots\}$, $\chi_\sigma=\{\tfrac1{16},\tfrac{17}{16},\ldots\}$ have nothing near $0.9$. Either a genuine non-Ising boundary feature of $p>0$ or a doubled-cluster convergence artifact; separable only with the cluster's converged $T$-ladders (NB7 'loose-end update').
- Cluster-sweep arbitration: $p=0.3/0.5$ multi-rung ratio tests, the Eq. (3) $B$–$C$ degeneracy at $T\le20$, and $T$-laddered partner spectroscopy at every $p$ (NB7's cluster-ready cells run automatically once `warm_sweep.jld2` lands).